In [ ]:
source dnanexus/bin/activate

In [ ]:
dx login --token X # replace with token
dx select "X" # replace with project

# 1: QC unphased array data

## 1.1: Get SNP list

In [ ]:
dx run app-swiss-army-knife --tag='SNP QC' --destination='DNM/snipar/QC/' -iin='/Bulk/Genotype Results/Genotype calls/ukb_snp_qc.txt' \
    -icmd="cat ukb_snp_qc.txt | awk '{ print \$1, \$159; }' > snps_prior_QC.txt && cat snps_prior_QC.txt | sed '1d' | awk '{ if (\$2 == 1) print \$1; }' > snps_post_QC.txt" \
    --instance-type mem1_ssd1_v2_x2 --priority low --brief -y --ignore-reuse

## 1.2: Get sample lists

In [ ]:
dx run app-swiss-army-knife --tag='Sample QC' --destination='DNM/snipar/QC/' -iin='Bulk/Genotype Results/Genotype calls/ukb_sqc_v2.txt' \
    -iin='Bulk/Genotype Results/Genotype calls/ukb22418_c1_b0_v2.fam' -icmd="cat ukb_sqc_v2.txt | cut -d ' ' -f 66 > in_phasing.txt && cat ukb22418_c1_b0_v2.fam | cut -d ' ' -f 1 > samples.tmp.txt && paste samples.tmp.txt in_phasing.txt > inds_prior_QC.txt && cat inds_prior_QC.txt | sed '1d' | awk '{ if (\$2 == 1) print \$1, \$1; }' > inds_post_QC.txt && rm samples.tmp.txt && rm in_phasing.txt" \
    --instance-type mem1_ssd1_v2_x4 --priority low --brief -y --ignore-reuse

## 1.3: Filter each chromosome

In [ ]:
for chr in {1..22}
do
    dx run app-swiss-army-knife --tag='Chrom QC' --destination='DNM/snipar/QC/' -iin="/Bulk/Genotype Results/Genotype calls/ukb22418_c${chr}_b0_v2.bed" \
        -iin="/Bulk/Genotype Results/Genotype calls/ukb22418_c${chr}_b0_v2.bim" -iin="/Bulk/Genotype Results/Genotype calls/ukb22418_c${chr}_b0_v2.fam" \
        -iin="/DNM/snipar/QC/snps_post_QC.txt" -iin="/DNM/snipar/QC/inds_post_QC.txt" -icmd="plink2 --bfile ukb22418_c${chr}_b0_v2 --extract snps_post_QC.txt --keep inds_post_QC.txt --export vcf bgz --out GT_chr${chr} && bcftools index GT_chr${chr}.vcf.gz" \
        --instance-type mem1_ssd1_v2_x2 --priority low --brief -y --ignore-reuse
done

# 2: Rename chromosomes in QC'ed unphased array data

## 2.1: Chromosome renaming between hg19 and hg38

In [ ]:
for chr in {1..22}
do
    echo "${chr} chr${chr}"
done > chr_rename.txt
dx upload chr_rename.txt --path='DNM/snipar/rename/'

## 2.2: Rename chromosomes

In [ ]:
ann="/mnt/project/DNM/snipar/rename/chr_rename.txt"

for chr in {1..22}
do
    vcff="/mnt/project/DNM/snipar/QC/GT_chr${chr}.vcf.gz"
    outf="GT_chr${chr}.b37.vcf.gz"
    dx run app-swiss-army-knife --tag='Chrom rename' --destination="DNM/snipar/rename/" -icmd="bcftools annotate -Oz -o $outf --rename-chrs $ann $vcff && bcftools index $outf" \
        --instance-type mem2_ssd1_v2_x2 --priority low --brief -y --ignore-reuse
done

# 3: Swap alleles in QC'ed unphased array data

In [ ]:
for chr in {1..22}
do
    vcff="/mnt/project/DNM/snipar/rename/GT_chr${chr}.b37.vcf.gz"
    outf="GT_chr${chr}.b37.swapped.vcf.gz"
    dx run app-swiss-army-knife --tag='Swap' --destination="DNM/snipar/swap/" -iimage_file="DNM/snipar/docker/swaprefalt_0.0.1.tar.gz" \
        -icmd="swapRefAlt_static --input $vcff --output $outf && bcftools index $outf" --instance-type mem2_ssd1_v2_x2 --priority low \
        --brief -y --ignore-reuse
done

# 4: Liftover coordinates in QC'ed unphased array data

## 4.1: Download the reference fasta

In [ ]:
wget "http://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/GRCh38_reference_genome/GRCh38_full_analysis_set_plus_decoy_hla.fa"
dx upload GRCh38_full_analysis_set_plus_decoy_hla.fa --path="DNM/snipar/liftover/"

## 4.2: Download the chain file

In [ ]:
wget "ftp://hgdownload.cse.ucsc.edu/goldenPath/hg19/liftOver/hg19ToHg38.over.chain.gz"
dx upload hg19ToHg38.over.chain.gz --path="DNM/snipar/liftover/"

## 4.3: Liftover coordinates

In [ ]:
for chr in {1..22}
do
    chain="/mnt/project/DNM/snipar/liftover/hg19ToHg38.over.chain.gz"
    vcff="/mnt/project/DNM/snipar/swap/GT_chr${chr}.b37.swapped.vcf.gz"
    ref="/mnt/project/DNM/snipar/liftover/GRCh38_full_analysis_set_plus_decoy_hla.fa"
    liff="GT_chr${chr}.b38.vcf.gz"
    sorf="GT_chr${chr}.b38.sorted.vcf.gz"
    
    dx run app-swiss-army-knife --tag='Liftover' --destination="DNM/snipar/liftover/" \
    -iimage_file="DNM/snipar/docker/liftovervcf_0.0.1.tar.gz" -icmd="liftoverVCF_static --input $vcff --output $liff --chain $chain --fasta $ref --chr chr$chr && bcftools sort -Oz -m 6G -o $sorf $liff && rm $liff && bcftools index $sorf" \
    --instance-type mem2_ssd1_v2_x2 --priority low --brief -y --ignore-reuse
done

# 5: Convert QC'ed unphased array data to BED

In [ ]:
for chr in {1..22}
do
    vcff="/mnt/project/DNM/snipar/liftover/GT_chr${chr}.b38.sorted.vcf.gz"

    dx run app-swiss-army-knife --tag='Convert to BED' --destination="DNM/snipar/GT/" -icmd="plink2 --vcf ${vcff} --id-delim --make-bed --out GT_chr${chr}" \
    --instance-type mem1_ssd1_v2_x16 --priority high --brief -y --ignore-reuse
done

# 6: Combine QC'ed unphased array data across chromosomes for KING

In [ ]:
for chr in {1..22}
do
    echo "DNM/snipar/GT/GT_chr${chr}" >> combined_paths.txt
    echo "GT_chr${chr}" >> combined_files.txt
done
dx upload combined_files.txt --path="DNM/snipar/GT/"

run="dx run swiss-army-knife --tag='Combine BED' --destination=\"DNM/snipar/GT\" -icmd=\"plink2 --pmerge-list /mnt/project/DNM/snipar/GT/combined_files.txt bfile --make-bed --out GT\" --instance-type \"mem1_ssd1_v2_x16\" --priority high --brief -y --ignore-reuse"

while IFS=' ' read -r path
do
    run+=" -iin=\"${path}.bed\""
    run+=" -iin=\"${path}.bim\""
    run+=" -iin=\"${path}.fam\""
done < "combined_paths.txt"

eval $run

# 7: Predict relationships based on QC'ed unphased array data and KING

In [ ]:
dx run swiss-army-knife --tag='KING' --destination="DNM/snipar" -iin="DNM/snipar/GT/GT.bed" -iin="DNM/snipar/GT/GT.bim" \
-iin="DNM/snipar/GT/GT.fam" -icmd="bash /mnt/project/scripts/king.sh" --instance-type "mem2_ssd1_v2_x32" --priority low --brief -y \
--ignore-reuse

# 8: Generate inds.txt, twins_pairs.txt, sibs_pairs.txt, etc. based on KING

In [ ]:
# inds.ipynb in JupyterLab

# 9: Predict IBD segments across sibs as determined by KING

In [ ]:
cmd="ibd.py --bed GT_chr@ --king king.kin0 --agesex sibs_agesex.txt --threads 60 --out chr@"
run="dx run swiss-army-knife --tag='snipar' --destination=\"DNM/snipar/IBD\" -iimage_file=\"DNM/snipar/docker/snipar.tar.gz\" -iin=\"DNM/snipar/king.kin0\" -iin=\"DNM/sibs_agesex.txt\" -icmd=\"${cmd}\" --instance-type \"mem2_ssd1_v2_x32\" --priority high --brief -y --ignore-reuse"

for chr in {1..22}
do 
    run+=" -iin=\"DNM/snipar/GT/GT_chr${chr}.bed\" -iin=\"DNM/snipar/GT/GT_chr${chr}.bim\" -iin=\"DNM/snipar/GT/GT_chr${chr}.fam\""
done

eval $run

# 10: Plot and trim IBD segments

In [ ]:
# snipar.ipynb in JupyterLab

# 11: Identify error profile

## 11.1: Generate PGEN files for WGS data

### 11.1.1: Generate PGEN files

In [ ]:
seq="Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]/"

for chr in {1..22}
do
    cmd="plink2 --pfile \"ukb24308_c${chr}_b0_v1\" --no-fam-pheno --keep \"/mnt/project/DNM/inds.txt\" --var-filter --make-pgen --out ukb_chr${chr}"
    dx run swiss-army-knife --tag='PGEN' --destination="DNM/PGEN (full)" -iin="${seq}ukb24308_c${chr}_b0_v1.pgen" \
    -iin="${seq}ukb24308_c${chr}_b0_v1.pvar" -iin="${seq}ukb24308_c${chr}_b0_v1.psam" -icmd="${cmd}" --instance-type "mem3_ssd3_x8" \
    --priority low --brief -y --ignore-reuse
done

## 11.2: Record MACs

In [ ]:
# unrelated.ipynb in JupyterLab

In [ ]:
seq="DNM/PGEN (full)/"

for chr in {1..22}
do
    cmd="plink2 --pfile \"ukb_chr${chr}\" --keep \"/mnt/project/DNM/unrelated_inds.txt\" --freq counts --out macs_chr${chr}"
    dx run swiss-army-knife --tag='MAC' --destination="DNM/MAC (full)" -iin="${seq}ukb_chr${chr}.pgen" -iin="${seq}ukb_chr${chr}.pvar" \
    -iin="${seq}ukb_chr${chr}.psam" -icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
done

## 11.3: Identify WGS mutation differences genome-wide

### 11.3.1: Identify WGS mutation differences

In [ ]:
for chr in {1..22}
do
    dx mkdir "DNM/twins/diffs (full)/chr${chr}"
done

seq="DNM/PGEN (full)/"

for chr in {1..22}
do
    for b in {0..1}
    do
        dx run swiss-army-knife --tag='Twins diffs' --destination="DNM/twins/diffs (full)/chr${chr}" -iin="${seq}ukb_chr${chr}.pgen" \
        -iin="${seq}ukb_chr${chr}.pvar" -iin="${seq}ukb_chr${chr}.psam" -icmd="bash /mnt/project/scripts/twins.sh ${chr} ${b}" \
        --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
    done
done

### 11.3.2: List MACs of mutation differences

In [ ]:
for chr in {1..22}
do
    cmd="papermill twins_full_list.ipynb output_twins_full_list.ipynb -p chrom ${chr} -p blocks 2"
    dx run dxjupyterlab --tag='Twins diffs list' --destination="DNM/twins/diffs (full)/chr${chr}" -iin="scripts/twins_full_list.ipynb" \
    -icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
done

## 11.4: Perform QC on mutation differences

In [ ]:
path="Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]"

for chr in {1..22}
do
    for b in {0..1}
    do
        cmd="papermill twins_full_qc.ipynb output_twins_full_qc.ipynb -p chrom ${chr} -p b ${b}"
        run="dx run dxjupyterlab --tag='Twins QC' --destination=\"DNM/twins/QC (full)\" -iduration=600 -iin=\"scripts/twins_full_qc.ipynb\" -icmd=\"${cmd}\" --instance-type \"mem3_ssd3_x4\" --priority high --brief -y --ignore-reuse"

        while IFS=' ' read -r iid
        do
            run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz\""
            run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz.tbi\""
        done < "blocks/twins_inds_b${b}.txt"

        eval $run
    done
done

## 11.5: QC and plot putative de novo mutations

In [ ]:
# twins_full_plot.ipynb in JupyterLab

# 12: Generate PGEN files for WGS data (filtered for GIAB)

## 12.1: Generate PGEN files

In [ ]:
seq="Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]/"

for chr in {1..22}
do
    cmd="plink2 --pfile \"ukb24308_c${chr}_b0_v1\" --no-fam-pheno --keep \"/mnt/project/DNM/inds.txt\" --snps-only --exclude bed1 \"/mnt/project/DNM/giab_difficult_merged_oct27.bed\" --var-filter --make-pgen --out ukb_chr${chr}"
    dx run swiss-army-knife --tag='PGEN' --destination="DNM/PGEN" -iin="${seq}ukb24308_c${chr}_b0_v1.pgen" \
    -iin="${seq}ukb24308_c${chr}_b0_v1.pvar" -iin="${seq}ukb24308_c${chr}_b0_v1.psam" -icmd="${cmd}" --instance-type "mem3_ssd3_x8" \
    --priority low --brief -y --ignore-reuse
done

## 12.2: Record MACs

In [ ]:
seq="DNM/PGEN/"

for chr in {1..22}
do
    cmd="plink2 --pfile \"ukb_chr${chr}\" --keep \"/mnt/project/DNM/unrelated_inds.txt\" --freq counts --out macs_chr${chr}"
    dx run swiss-army-knife --tag='MAC' --destination="DNM/MAC" -iin="${seq}ukb_chr${chr}.pgen" -iin="${seq}ukb_chr${chr}.pvar" \
    -iin="${seq}ukb_chr${chr}.psam" -icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
done

# 13: Call de novo mutations in trios

## 13.1: Identify WGS mutation differences genome-wide

### 13.1.1: Identify WGS mutation differences

In [ ]:
for chr in {1..22}
do
    dx mkdir "DNM/trios/diffs/chr${chr}"
done

In [ ]:
seq="DNM/PGEN/"

for chr in {1..22}
do
    for b in {0..10}
    do
        dx run swiss-army-knife --tag='Trios diffs' --destination="DNM/trios/diffs/chr${chr}" -iin="${seq}ukb_chr${chr}.pgen" \
        -iin="${seq}ukb_chr${chr}.pvar" -iin="${seq}ukb_chr${chr}.psam" -icmd="bash /mnt/project/scripts/trios.sh ${chr} ${b}" \
        --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
    done
done

### 13.1.2: List MACs of mutation differences

In [ ]:
for chr in {1..22}
do
    cmd="papermill trios_list.ipynb output_trios_list.ipynb -p chrom ${chr} -p blocks 11"
    dx run dxjupyterlab --tag='Trios diffs list' --destination="DNM/trios/diffs/chr${chr}" -iin="scripts/trios_list.ipynb" \
    -icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
done

## 13.2: Perform QC on mutation differences

In [ ]:
path="Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]"

for b in {0..10}
do
    cmd="papermill trios_qc.ipynb output_trios_qc.ipynb -p b ${b}"
    run="dx run dxjupyterlab --tag='Trios QC' --destination=\"DNM/trios/QC\" -iduration=600 -iin=\"scripts/trios_qc.ipynb\" -icmd=\"${cmd}\" --instance-type \"mem3_ssd3_x4\" --priority high --brief -y --ignore-reuse"

    while IFS=' ' read -r iid
    do
        run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz\""
        run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz.tbi\""
    done < "blocks/trios_inds_b${b}.txt"

    eval $run
done

## 13.3: QC and plot putative de novo mutations

In [ ]:
# trios_plot.ipynb in JupyterLab

## 12.4: Predict mutational consequences

In [ ]:
cmd="papermill trios_vep.ipynb output_trios_vep.ipynb"
dx run dxjupyterlab_spark_cluster --tag='Consequences in trios' --destination="DNM/trios/out" -iin="scripts/trios_vep.ipynb" \
-icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" --instance-count=2 -ifeature="HAIL-VEP" --priority high --brief -y --ignore-reuse

In [ ]:
# trios_consequences.ipynb in JupyterLab

## 13.1: Identify WGS mutation differences genome-wide

### 13.1.1: Identify WGS mutation differences

In [ ]:
for chr in {1..22}
do
    dx mkdir "DNM/twins/diffs/chr${chr}"
done

In [ ]:
seq="DNM/PGEN/"

for chr in {1..22}
do
    for b in {0..1}
    do
        dx run swiss-army-knife --tag='Twins diffs' --destination="DNM/twins/diffs/chr${chr}" -iin="${seq}ukb_chr${chr}.pgen" \
        -iin="${seq}ukb_chr${chr}.pvar" -iin="${seq}ukb_chr${chr}.psam" -icmd="bash /mnt/project/scripts/twins.sh ${chr} ${b}" \
        --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
    done
done

### 13.1.2: List MACs of mutation differences

In [ ]:
for chr in {1..22}
do
    cmd="papermill twins_list.ipynb output_twins_list.ipynb -p chrom ${chr} -p blocks 2"
    dx run dxjupyterlab --tag='Twins diffs list' --destination="DNM/twins/diffs/chr${chr}" -iin="scripts/twins_list.ipynb" \
    -icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
done

## 13.2: Perform QC on mutation differences

In [ ]:
path="Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]"

for b in {0..1}
do
    cmd="papermill twins_qc.ipynb output_twins_qc.ipynb -p b ${b}"
    run="dx run dxjupyterlab --tag='Twins QC' --destination=\"DNM/twins/QC\" -iduration=600 -iin=\"scripts/twins_qc.ipynb\" -icmd=\"${cmd}\" --instance-type \"mem3_ssd3_x4\" --priority high --brief -y --ignore-reuse"

    while IFS=' ' read -r iid
    do
        run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz\""
        run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz.tbi\""
    done < "blocks/twins_inds_b${b}.txt"

    eval $run
done

## 13.3: QC and plot putative early developmental mutations

In [ ]:
# twins_plot.ipynb in JupyterLab

# 14: Call de novo mutations in sibs

## 14.1: Identify WGS mutation differences genome-wide

### 14.1.1: Identify WGS mutation differences

In [ ]:
for chr in {1..22}
do
    dx mkdir "DNM/sibs/diffs/chr${chr}"
done

In [ ]:
seq="DNM/PGEN/"

for chr in {1..22}
do
    for b in {0..223}
    do
        dx run swiss-army-knife --tag='Sibs diffs updated' --destination="DNM/sibs/diffs/chr${chr}" -iin="${seq}ukb_chr${chr}.pgen" \
        -iin="${seq}ukb_chr${chr}.pvar" -iin="${seq}ukb_chr${chr}.psam" -icmd="bash /mnt/project/scripts/sibs.sh ${chr} ${b}" \
        --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
    done
done

### 14.1.2: List MACs of mutation differences

In [ ]:
for chr in {1..22}
do
    cmd="papermill sibs_list.ipynb output_sibs_list.ipynb -p chrom ${chr} -p blocks 224"
    dx run dxjupyterlab --tag='Sibs diffs list' --destination="DNM/sibs/diffs/chr${chr}" -iin="scripts/sibs_list.ipynb" \
    -icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
done

## 14.2: Perform QC on mutation differences

In [ ]:
path="Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]"

for b in {0..223}
do
    cmd="papermill sibs_qc.ipynb output_sibs_qc.ipynb -p b ${b}"
    run="dx run dxjupyterlab --tag='Sibs QC' --destination=\"DNM/sibs/QC\" -iduration=600 -iin=\"scripts/sibs_qc.ipynb\"  -icmd=\"${cmd}\" --instance-type \"mem3_ssd3_x4\" --priority high --brief -y --ignore-reuse"

    while IFS=' ' read -r iid
    do
        run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz\""
        run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz.tbi\""
    done < "blocks/sibs_inds_b${b}.txt"

    eval $run
done

## 14.3: QC and plot putative de novo mutations

In [ ]:
# sibs_plot.ipynb in JupyterLab

## 14.4: Predict mutational consequences

In [ ]:
cmd="papermill sibs_vep.ipynb output_sibs_vep.ipynb"
dx run dxjupyterlab_spark_cluster --tag='Consequences in sibs' --destination="DNM/sibs/out" -iin="scripts/sibs_vep.ipynb" \
-icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" --instance-count=2 -ifeature="HAIL-VEP" --priority high --brief -y --ignore-reuse

In [ ]:
# sibs_consequences.ipynb in JupyterLab

# 15: Transmition validation

In [ ]:
path="Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]"

for b in {0..3}
do
    path="Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]"

    cmd="papermill duos_qc.ipynb output_duos_qc.ipynb -p b ${b}"
    run="dx run dxjupyterlab --tag='Duos QC' --destination=\"DNM/validation/\" -iduration=600 -iin=\"scripts/duos_qc.ipynb\" -icmd=\"${cmd}\" --instance-type \"mem3_ssd3_x4\" --priority high --brief -y --ignore-reuse"
    
    while IFS=' ' read -r iid
    do
        run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz\""
        run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz.tbi\""
    done < "blocks/duo_inds_b${b}.txt"
    
    eval $run
done

In [ ]:
path="Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]"

for b in {0..4}
do
    path="Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]"

    cmd="papermill o_qc.ipynb output_o_qc.ipynb -p b ${b}"
    run="dx run dxjupyterlab --tag='Offspring QC' --destination=\"DNM/validation/\" -iduration=600 -iin=\"scripts/o_qc.ipynb\" -icmd=\"${cmd}\" --instance-type \"mem3_ssd3_x4\" --priority high --brief -y --ignore-reuse"
    
    while IFS=' ' read -r iid
    do
        run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz\""
        run+=" -iin=\"${path}/${iid:0:2}/${iid}_24051_0_0.dragen.hard-filtered.gvcf.gz.tbi\""
    done < "blocks/o_inds_b${b}.txt"
    
    eval $run
done

In [ ]:
# transmission.ipynb in JupyterLab

# 16: Variants in DNA repair genes across parents of trios and sibs

## 16.1: Extract variants in DNA repair genes

In [ ]:
seq="DNM/PGEN (full)/"

for chr in {1..20} 22
do
    cmd="plink2 --pfile \"ukb_chr${chr}\" --extract bed1 \"/mnt/project/DNA repair genes/repair_genes_map.bed\" --geno 0.01 --no-fam-pheno --make-pgen --out ukb_chr${chr}"
    dx run swiss-army-knife --tag='Variants in DNA repair genes' --destination="DNA repair genes/PGEN" \
    -iin="${seq}ukb_chr${chr}.pgen" -iin="${seq}ukb_chr${chr}.pvar" -iin="${seq}ukb_chr${chr}.psam" \
    -icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
done

## 16.2: Predict consequences of variants in DNA repair genes

In [ ]:
# vcf.ipynb in JupyterLab

In [ ]:
for chr in {1..20} 22
do
    cmd="papermill vep.ipynb output_vep_chr${chr}.ipynb -p chrom ${chr}"
    dx run dxjupyterlab_spark_cluster --tag='Consequences of variants in DNA repair genes' \
    --destination="DNA repair genes/VEP" -iin="scripts/vep.ipynb" -icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" \
    --instance-count=2 -ifeature="HAIL-VEP" --priority high --brief -y --ignore-reuse
done

In [ ]:
# consequences.ipynb in JupyterLab

## 16.3: List LoF and missense variants

In [ ]:
seq="DNA repair genes/PGEN/"

for chr in {1..20} 22
do
    dx run swiss-army-knife --tag='List LoF variants in DNA repair genes across parents and sibs' \
    --destination="DNA repair genes/counts" -iin="${seq}ukb_chr${chr}.pgen" -iin="${seq}ukb_chr${chr}.pvar" \
    -iin="${seq}ukb_chr${chr}.psam" -icmd="bash /mnt/project/scripts/lofs.sh ${chr}" \
    --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse

     dx run swiss-army-knife --tag='List missense variants in DNA repair genes across parents and sibs' \
    --destination="DNA repair genes/counts" -iin="${seq}ukb_chr${chr}.pgen" -iin="${seq}ukb_chr${chr}.pvar" \
    -iin="${seq}ukb_chr${chr}.psam" -icmd="bash /mnt/project/scripts/missense.sh ${chr}" \
    --instance-type "mem1_ssd1_v2_x16" --priority high --brief -y --ignore-reuse
done

## 16.4: Record MACs of LoF and missense variants

In [ ]:
seq="DNA repair genes/PGEN/"

for chr in {1..20} 22
do
    cmd="plink2 --pfile \"ukb_chr${chr}\" --extract \"/mnt/project/DNA repair genes/repair_genes_lofs.txt\" --freq counts --out repair_genes_lofs_chr${chr}"
    dx run swiss-army-knife --tag='MAC' --destination="DNA repair genes/MAC" -iin="${seq}ukb_chr${chr}.pgen" \
    -iin="${seq}ukb_chr${chr}.pvar" -iin="${seq}ukb_chr${chr}.psam" -icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" \
    --priority high --brief -y --ignore-reuse

    cmd="plink2 --pfile \"ukb_chr${chr}\" --extract \"/mnt/project/DNA repair genes/repair_genes_miss.txt\" --freq counts --out repair_genes_miss_chr${chr}"
    dx run swiss-army-knife --tag='MAC' --destination="DNA repair genes/MAC" -iin="${seq}ukb_chr${chr}.pgen" \
    -iin="${seq}ukb_chr${chr}.pvar" -iin="${seq}ukb_chr${chr}.psam" -icmd="${cmd}" --instance-type "mem1_ssd1_v2_x16" \
    --priority high --brief -y --ignore-reuse
done

In [ ]:
# families.ibynb in JupyterLab

In [ ]:
# genotypes.ibynb in JupyterLab

## 16.5: Burden tests

In [ ]:
# rates.ipynb in JupyterLab

In [ ]:
# ukb_data.ipynb in JupyterLab

In [ ]:
# gene_giab.ipynb in JupyterLab

In [ ]:
# burden.ipynb in JupyterLab